# D2b CNN 아키텍처와 CIFAR-10 — 실습 (W5)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> Colab에서 **런타임 → GPU** 권장(CPU도 동작, 학습에 몇 분).
> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. SmallCNN을 정의하고 **shape 깔때기**를 층마다 추적한다
2. 파라미터 **268,650개**를 검산한다 (98%가 FC!)
3. **D1c 학습 루프를 그대로 재사용**해 CIFAR-10을 학습한다
4. **MLP vs CNN 정면 대결** — 같은 조건에서 승부를 확인한다
5. 클래스별 정확도로 "누가 쉽고 누가 어려운가"를 진단한다

**7단계 멘탈모델 초점:** 표현 + 모델

> ⏱️ 시간을 위해 **부분집합**(학습 1만 장·평가 2천 장)·3 epoch — 목적은 절대 성능이 아니라 **비교와 경향**.

## Part A. CIFAR-10 로드 — 배치 shape 확인

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈
import matplotlib.pyplot as plt                         # 그래프
from torchvision import datasets, transforms            # 영상 데이터
from torch.utils.data import DataLoader, Subset         # 로더·부분집합

device = 'cuda' if torch.cuda.is_available() else 'cpu' # 장치
torch.manual_seed(0)                                    # 재현성
tf = transforms.ToTensor()                              # 0~1 텐서
train_ds = datasets.CIFAR10('./data', train=True,  download=True, transform=tf)  # 5만 장
test_ds  = datasets.CIFAR10('./data', train=False, download=True, transform=tf)  # 1만 장
classes = train_ds.classes                              # 클래스 10종

train_loader = DataLoader(Subset(train_ds, range(10000)), batch_size=64, shuffle=True)  # 부분집합 1만
test_loader  = DataLoader(Subset(test_ds,  range(2000)),  batch_size=256)               # 평가 2천
xb, yb = next(iter(train_loader))                       # 배치 하나
print('배치 shape:', xb.shape)                          # (64,3,32,32) — 이제 채널 3(컬러)!
print('클래스:', classes)                               # airplane ~ truck

## Part B. SmallCNN 정의 — shape 깔때기 ⭐
[Conv→ReLU→Pool] 2블록 → Flatten → FC. **각 층의 출력 shape을 먼저 예측**하고 코드로 확인하세요.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(                  # 특징 추출부
            nn.Conv2d(3, 16, 3, padding=___), nn.ReLU(),  # ✍️ 빈칸: 크기 유지 패딩 → (16,32,32)
            nn.MaxPool2d(2),                            # (16,16,16) — 절반
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), # (32,16,16) — 지도 32장
            nn.MaxPool2d(___),                          # ✍️ 빈칸: 다시 절반 → (32,8,8)
        )
        self.classifier = nn.Sequential(                # 분류부 = D1c의 MLP!
            nn.Flatten(),                               # (32,8,8) → 2048
            nn.Linear(32 * 8 * ___, 128), nn.ReLU(),    # ✍️ 빈칸: 풀링 2번 후 한 변 크기
            nn.Linear(128, 10)                          # logit 10개 (softmax X — D1b)
        )
    def forward(self, x):
        return self.classifier(self.features(x))        # 특징 추출 → 분류

cnn = SmallCNN().to(device)                             # 생성
z = torch.rand(1, 3, 32, 32).to(device)                 # 시험 입력
print('features 출력:', cnn.features(z).shape)          # (1,32,8,8) — 깔때기 확인
n_cnn = sum(p.numel() for p in cnn.parameters())        # 파라미터 검산(D1c)
print('CNN 파라미터:', n_cnn)                           # 268,650 — 표와 일치?
n_fc = sum(p.numel() for p in cnn.classifier.parameters())  # 분류부만
print('그중 FC(분류부):', n_fc, '=', round(100 * n_fc / n_cnn), '%')  # ~98% — 공유의 반증

## Part C. 학습 — D1c 루프 재사용
**글자 하나 안 바뀝니다.** 모델만 CNN으로 교체. (부분집합·3 epoch)

In [ ]:
def train_model(model, epochs=3):                       # 재사용 가능한 학습 함수(D1c 루프)
    criterion = nn.___()                                # ✍️ 빈칸: 분류 손실(logit 입력)
    optimizer = torch.optim.___(model.parameters(), lr=1e-3)  # ✍️ 빈칸: 적응형 옵티마이저(D1b)
    losses = []
    for epoch in range(epochs):
        model.train()                                   # 학습 모드
        running = 0.0
        for xb, yb in train_loader:                     # ① 배치
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()                       # ② 초기화
            pred = model(xb)                            # ③ 순전파
            loss = criterion(pred, yb)                  # ④ 손실
            loss.backward()                             # ⑤a 역전파
            optimizer.step()                            # ⑤b 갱신
            running += loss.item()
        losses.append(running / len(train_loader))      # epoch 평균
        print(f'  epoch {epoch+1}: train loss = {losses[-1]:.4f}')
    return losses

def accuracy(model):                                    # 평가 함수(D1c 그대로)
    model.eval()                                        # 평가 모드
    correct = total = 0
    with torch.no_grad():                               # 기록 끄기
        for xb, yb in test_loader:
            pred = model(xb.to(device)).argmax(___).cpu()  # ✍️ 빈칸: 클래스 차원(D1a)
            correct += (pred == yb).sum().item(); total += yb.size(0)
    return correct / total

print('CNN 학습:')
cnn_losses = train_model(cnn)                           # 3 epoch
cnn_acc = accuracy(cnn)                                 # 평가
print('CNN test accuracy:', round(cnn_acc, 3))          # 무작위(0.1)의 5배 이상

## Part D. 정면 대결 — MLP 선수 입장 ⭐⭐
같은 데이터·epoch·옵티마이저. MLP는 **6배 큰 파라미터**를 들고 나옵니다.

In [ ]:
mlp = nn.Sequential(                                    # MLP 선수(D1c 스타일)
    nn.___(),                                           # ✍️ 빈칸: (3,32,32)를 한 줄로 펴는 층
    nn.Linear(3072, 512), nn.ReLU(),                    # 3072 = 32*32*3
    nn.Linear(512, 10)                                  # logit
).to(device)
n_mlp = sum(p.numel() for p in mlp.parameters())        # 파라미터
print('MLP 파라미터:', n_mlp, '(CNN의', round(n_mlp / n_cnn, 1), '배)')  # ~1,578,506

print('MLP 학습:')
mlp_losses = train_model(mlp)                           # 같은 함수 = 같은 조건
mlp_acc = accuracy(mlp)                                 # 같은 평가
print('MLP test accuracy:', round(mlp_acc, 3))
print()
print(f'대결 결과 — CNN {cnn_acc:.3f} vs MLP {mlp_acc:.3f} → 승자: ' + ('CNN' if cnn_acc > mlp_acc else 'MLP'))
print('구조(27만)가 용량(157만)을 이겼는가?', cnn_acc > mlp_acc)  # 구조 > 용량

In [ ]:
plt.figure(figsize=(6, 4))                              # 손실 곡선 비교
plt.plot(range(1, 4), cnn_losses, 'o-', label=f'CNN ({n_cnn:,} params)')   # CNN
plt.plot(range(1, 4), mlp_losses, 's--', label=f'MLP ({n_mlp:,} params)')  # MLP
plt.xlabel('epoch'); plt.ylabel('train loss')           # 축(영어)
plt.xticks([1, 2, 3])                                   # 정수 눈금
plt.title('CNN vs MLP on CIFAR-10 (same setup)')        # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # 작은 CNN이 더 낮은 손실로

## Part E. 진단 — 클래스별 정확도
누가 쉽고 누가 어려운가. MNIST의 "모양 닮음"(4↔9) 대신 "**의미 닮음**"(고양이↔개)이 나타납니다.

In [ ]:
correct_c = torch.zeros(10); total_c = torch.zeros(10)  # 클래스별 집계
cnn.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        preds = cnn(xb.to(device)).argmax(1).cpu()      # CNN 예측
        for t, p in zip(yb, preds):
            total_c[t] += 1                             # 그 클래스 출제 수
            if t == p: correct_c[t] += 1                # 맞힌 수
acc_c = correct_c / total_c                             # 클래스별 정확도

order = acc_c.argsort(descending=True)                  # 잘하는 순 정렬
plt.figure(figsize=(7, 4))                              # 막대그래프
plt.bar([classes[i] for i in order], acc_c[order])      # 클래스명(영어라 폰트 OK)
plt.xticks(rotation=45, ha='right')                     # 라벨 회전
plt.ylabel('accuracy'); plt.title('Per-class accuracy (CNN)')  # 축·제목(영어)
plt.grid(True, axis='y'); plt.tight_layout(); plt.show()  # 쉬운/어려운 클래스
for i in order:                                         # 표로도 출력
    print(f'{classes[i]:<11}: {acc_c[i]:.2f}')          # 진단: 어디가 약한가

> **배경이 단순한 ship이 압도적으로 높고**, **동물들**(dog·horse·cat·bird)이 바닥권 — 포즈·색이 다양하고 서로 닮았기 때문. 더 잘하려면? ①더 깊은 모델 ②더 많은 데이터·epoch ③**이미 잘 배운 모델 빌리기 — 다음 주 전이학습.**

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "Conv 블록을 하나 더 쌓으면(32→64, pool) Flatten 입력이 몇이 되는지 내가 계산할 테니 채점해 줘." (답: 64×4×4=1024)
- "MLP가 파라미터는 6배인데 왜 지는지, '구조'라는 말로 내가 설명해 볼게."
- "클래스별 정확도에서 cat이 낮은 이유 가설 3개를 내가 낼 테니 검증 방법을 물어봐 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. SmallCNN의 shape 깔때기를 추적하고 파라미터 268,650(98%가 FC)을 검산했다
2. D1c 학습 루프를 그대로 재사용해 CIFAR-10을 학습했다
3. 같은 조건 대결에서 6배 작은 CNN이 MLP를 이김 — **구조가 용량을 이긴다**

**스스로 점검**
- [ ] Linear(2048, 128)의 2048이 어디서 왔는지 안다(32×8×8)
- [ ] 파라미터가 FC에 몰리는 이유를 가중치 공유로 설명할 수 있다
- [ ] 대결 실험이 왜 '같은 조건'이어야 공정한지 안다
- [ ] 클래스별 정확도에서 의미 닮음 패턴을 읽을 수 있다

**🔹심화 (선택)**
- SmallCNN에 **세 번째 블록**(Conv 32→64 + Pool)을 추가해 보세요. Flatten 입력(64×4×4=1024)을 먼저 계산하고, 정확도·파라미터 변화를 표로 정리.
- 첫 Conv의 학습된 필터를 시각화해 보세요: `w = cnn.features[0].weight.detach().cpu()` → 16개를 imshow. 엣지/색 검출기가 보이나요? (D2a "필터는 학습된다"의 증거)
- EPOCHS를 10으로 늘려 train 손실과 test 정확도의 관계(과적합 신호)를 관찰하세요.